## workflow: Build a CNN-LSTM hybrid model to classify the customer product reviews into good or bad

In [10]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/raw/product_reviews_full_dataset.csv')

# Display the first few rows and info to understand the data
print("DataFrame head:")
display(df.head())
print("\nDataFrame info:")
df.info()

DataFrame head:


,id,brand,categories,dateAdded,dateUpdated,ean,keys,manufacturer,manufacturerNumber,name,...,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,upc
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",2017-07-25T00:52:42Z,2018-02-05T08:36:58Z,6.02537E+11,"602537205981,universalmusic/14331328,universal...",Universal Music Group / Cash Money,14331328,Pink Friday: Roman Reloaded Re-Up (w/dvd),...,NaN,0.0,5,https://redsky.target.com/groot-domain-api/v1/...,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,Joshua,6.02537E+11
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",2017-07-25T05:16:03Z,2018-02-05T11:27:45Z,73416000391,lundbergorganiccinnamontoastricecakes/b000fvzw...,Lundberg,574764,Lundberg Organic Cinnamon Toast Rice Cakes,...,100209113.0,NaN,5,https://www.walmart.com/reviews/product/29775278,Good flavor. This review was collected as part...,Good,NaN,NaN,Dorothy W,73416000391
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",2017-07-25T05:16:03Z,2018-02-05T11:27:45Z,73416000391,lundbergorganiccinnamontoastricecakes/b000fvzw...,Lundberg,574764,Lundberg Organic Cinnamon Toast Rice Cakes,...,100209113.0,NaN,5,https://www.walmart.com/reviews/product/29775278,Good flavor.,Good,NaN,NaN,Dorothy W,73416000391
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",2017-07-25T16:26:19Z,2018-02-05T11:25:51Z,67981934427,"kylovesensualitypleasuregel/b00u2whx8s,0679819...",K-Y,67981934427,K-Y Love Sensuality Pleasure Gel,...,113026909.0,NaN,1,https://www.walmart.com/reviews/product/43383370,I read through the reviews on here before look...,Disappointed,NaN,NaN,Rebecca,67981934427
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",2017-07-25T16:26:19Z,2018-02-05T11:25:51Z,67981934427,"kylovesensualitypleasuregel/b00u2whx8s,0679819...",K-Y,67981934427,K-Y Love Sensuality Pleasure Gel,...,171267657.0,NaN,1,https://www.walmart.com/reviews/product/43383370,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,Walker557,67981934427



DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71044 entries, 0 to 71043
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    71044 non-null  object 
 1   brand                 71044 non-null  object 
 2   categories            71044 non-null  object 
 3   dateAdded             71044 non-null  object 
 4   dateUpdated           71044 non-null  object 
 5   ean                   39065 non-null  object 
 6   keys                  71044 non-null  object 
 7   manufacturer          70903 non-null  object 
 8   manufacturerNumber    70841 non-null  object 
 9   name                  71044 non-null  object 
 10  reviews.date          70977 non-null  object 
 11  reviews.dateAdded     71044 non-null  object 
 12  reviews.dateSeen      71044 non-null  object 
 13  reviews.didPurchase   32158 non-null  object 
 14  reviews.doRecommend   60429 non-null  object 
 15  re

### Create Target Variable

As per the workflow, a `target` feature is created. If a customer is pleased by the product, the rating is higher than 3. Any rating below 4 shows that the customer doesn’t like the product. This means `reviews.rating < 4` will be `True` for 'bad' reviews and `False` for 'good' reviews.

In [11]:
# Create the target feature
df['target'] = df['reviews.rating'] < 4

# Display value counts of the target variable
print("Target variable distribution:")
display(df['target'].value_counts())

Target variable distribution:


,count
target,
False,61141
True,9903


### Prepare X and Y and Split Data

Now, we'll create `X` with `reviews.text` and `Y` with the `target` column. Then, we'll split the dataset into training and testing sets in an 80:20 ratio.

In [14]:
from sklearn.model_selection import train_test_split

# Drop rows where 'reviews.text' or 'reviews.rating' are missing
df.dropna(subset=['reviews.text', 'reviews.rating'], inplace=True)

# Create X and Y
X = df['reviews.text']
Y = df['target']

# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (56806,)
X_test shape: (14202,)
y_train shape: (56806,)
y_test shape: (14202,)


### Text Vectorization and Padding

Now, we'll use Keras's `Tokenizer` to convert the text reviews into sequences of integers. We'll limit the vocabulary to `MAX_NB_WORDS = 20000` and pad the sequences to `MAX_SEQUENCE_LENGTH = 150`.

In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

MAX_NB_WORDS = 20000
MAX_SEQUENCE_LENGTH = 150

# Initialize and fit the tokenizer on the training data
tokenizer = Tokenizer(num_words=MAX_NB_WORDS)
tokenizer.fit_on_texts(X_train)

# Convert text to sequences for train and test sets
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

# Pad sequences to a fixed length
x_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_SEQUENCE_LENGTH)
x_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_SEQUENCE_LENGTH)

# One-hot encode the output classes
y_train_one_hot = to_categorical(y_train, num_classes=2)
y_test_one_hot = to_categorical(y_test, num_classes=2)

print(f"Shape of x_train_padded: {x_train_padded.shape}")
print(f"Shape of x_test_padded: {x_test_padded.shape}")
print(f"Shape of y_train_one_hot: {y_train_one_hot.shape}")
print(f"Shape of y_test_one_hot: {y_test_one_hot.shape}")

Shape of x_train_padded: (56806, 150)
Shape of x_test_padded: (14202, 150)
Shape of y_train_one_hot: (56806, 2)
Shape of y_test_one_hot: (14202, 2)


### Build and Train the CNN-LSTM Hybrid Model

Now, I'll construct the CNN-LSTM model using Keras, including the Embedding, Conv1D, MaxPooling1D, Dropout, LSTM, and Dense layers as specified in the workflow. After defining the architecture, I'll compile the model, train it for 5 epochs, and then evaluate its performance on the test set.

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout

# Define the CNN-LSTM model
model = Sequential()
model.add(Embedding(input_dim=MAX_NB_WORDS, output_dim=50, input_length=MAX_SEQUENCE_LENGTH))
model.add(Conv1D(filters=64, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=5))
model.add(Dropout(0.2))
model.add(Conv1D(filters=64, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=5))
model.add(Dropout(0.2))
model.add(LSTM(64))
model.add(Dense(2, activation='softmax')) # 2 neurons for good/bad classification

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
print("Model Summary:")
model.summary()

# Train the model
print("\nTraining the model...")
history = model.fit(
    x_train_padded, y_train_one_hot,
    epochs=5,
    batch_size=64,
    validation_split=0.1 # Using a small validation split during training
)

# Evaluate the model on the test data
print("\nEvaluating the model on test data...")
loss, accuracy = model.evaluate(x_test_padded, y_test_one_hot, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Model Summary:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the model...
Epoch 1/5
799/799 ━━━━━━━━━━━━━━━━━━━━ 51s 59ms/step - accuracy: 0.8916 - loss: 0.2824 - val_accuracy: 0.9090 - val_loss: 0.2249
Epoch 2/5
799/799 ━━━━━━━━━━━━━━━━━━━━ 43s 54ms/step - accuracy: 0.9277 - loss: 0.1903 - val_accuracy: 0.9155 - val_loss: 0.2157
Epoch 3/5
799/799 ━━━━━━━━━━━━━━━━━━━━ 46s 57ms/step - accuracy: 0.9476 - loss: 0.1455 - val_accuracy: 0.9132 - val_loss: 0.2391
Epoch 4/5
799/799 ━━━━━━━━━━━━━━━━━━━━ 44s 55ms/step - accuracy: 0.9634 - loss: 0.1082 - val_accuracy: 0.9095 - val_loss: 0.2543
Epoch 5/5
799/799 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - accuracy: 0.9740 - loss: 0.0789 - val_accuracy: 0.9146 - val_loss: 0.2768

Evaluating the model on test data...
Test Loss: 0.2578
Test Accuracy: 0.9204


### Predict on New Review Data

Let's create a small dataset of new reviews and use the trained model to predict their sentiment (good or bad).

In [17]:
import numpy as np

# Create a sample of new reviews
new_reviews = [
    "This product is absolutely amazing! I love it so much.",
    "Not happy with this purchase. It broke after one use.",
    "It's okay, nothing special. I might buy it again, I might not.",
    "Worst product ever! Complete waste of money and time.",
    "Highly recommend! Exceeded all my expectations."
]

# Preprocess the new reviews using the same tokenizer and padding
new_reviews_sequences = tokenizer.texts_to_sequences(new_reviews)
new_reviews_padded = pad_sequences(new_reviews_sequences, maxlen=MAX_SEQUENCE_LENGTH)

# Make predictions
predictions = model.predict(new_reviews_padded)

# Interpret the predictions
predicted_classes = np.argmax(predictions, axis=1)

# Map numerical predictions back to 'Good' (False) or 'Bad' (True) sentiment
sentiment_map = {0: 'Good Review', 1: 'Bad Review'}
predicted_sentiments = [sentiment_map[p] for p in predicted_classes]

# Display the original reviews and their predicted sentiments
print("--- New Review Predictions ---")
for i, review in enumerate(new_reviews):
    print(f"Review: '{review}'\nPredicted Sentiment: {predicted_sentiments[i]}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step
--- New Review Predictions ---
Review: 'This product is absolutely amazing! I love it so much.'
Predicted Sentiment: Good Review

Review: 'Not happy with this purchase. It broke after one use.'
Predicted Sentiment: Bad Review

Review: 'It's okay, nothing special. I might buy it again, I might not.'
Predicted Sentiment: Bad Review

Review: 'Worst product ever! Complete waste of money and time.'
Predicted Sentiment: Bad Review

Review: 'Highly recommend! Exceeded all my expectations.'
Predicted Sentiment: Good Review

